# Tarea 2.2 — Preprocesamiento del Proyecto Telco Churn

**Curso:** MAI 540 — Módulo 2 · **Proyecto:** predicción de fuga de clientes (churn)

Este cuaderno construye la herramienta de preprocesamiento gobernada por `context.md` (Tarea 2.1). Cubre, en este orden:

1. Diagnóstico del estado del conjunto de datos y tratamiento de valores faltantes.
2. Duplicados y valores atípicos, con la evidencia detrás de cada decisión.
3. Codificación de variables categóricas y escalado de numéricas.
4. Selección de características.
5. El orden completo está diseñado para que **ningún estadístico usado por el pipeline se calcule con datos del conjunto de prueba** — la justificación de cada paso está en su celda de texto.

No se sube el CSV a este repositorio (restricción de `context.md`, sección "Datos personales"): la celda siguiente lo pide por el diálogo de carga de Colab.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

## Paso 0 — Carga de datos

Si este cuaderno corre en Google Colab, pide el archivo por el diálogo de carga (`files.upload()`). Si corre localmente, ajusta `DATA_PATH` a la ruta del CSV en tu equipo. En ningún caso el CSV se versiona en este repositorio.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import files
    print("Sube el archivo 'datos Proyecto - Telco_Churn.csv' en el dialogo que se abre a continuacion...")
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]
else:
    DATA_PATH = "datos Proyecto - Telco_Churn.csv"  # ajusta la ruta si ejecutas localmente

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## Paso 1 — Eliminar variables prohibidas (`context.md`)

`TotalCharges` (fuga de información), `customerID` (identificador directo) y `gender` (atributo protegido sin poder predictivo) se eliminan **antes de cualquier otra transformación**, tal como exige `context.md`.

In [ ]:
PROHIBIDAS = ["TotalCharges", "customerID", "gender"]
df = df.drop(columns=PROHIBIDAS)
print(f"Columnas tras eliminar prohibidas: {df.shape[1]}")

## Paso 2 — Recodificación de categorías redundantes (R2/R3 de `context.md`)

`"No internet service"` y `"No phone service"` ya están implícitas en `InternetService = No` y `PhoneService = No` respectivamente. Se recodifican a `"No"` para no duplicar la misma señal en varias columnas. Es una regla fija (no aprende nada de los datos), por lo que es segura de aplicar antes de particionar.

In [ ]:
SERVICIOS_INTERNET = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                       "TechSupport", "StreamingTV", "StreamingMovies"]
for c in SERVICIOS_INTERNET:
    df[c] = df[c].replace("No internet service", "No")
df["MultipleLines"] = df["MultipleLines"].replace("No phone service", "No")
print("Recodificado. Valores unicos de OnlineSecurity ahora:", df["OnlineSecurity"].unique())

## Paso 3 — Diagnóstico del estado del conjunto de datos

Reporte descriptivo (conteos, no estadísticos de un modelo) sobre el conjunto completo: nulos declarados, cadenas vacías, duplicados y valores atípicos. Como no calcula ni guarda ningún parámetro para el pipeline, es seguro hacerlo antes de particionar — es solo lectura.

In [ ]:
print("=== Nulos declarados (NaN) por columna ===")
nulos = df.isnull().sum()
print(nulos[nulos > 0] if (nulos > 0).any() else "Ninguno.")

print("\n=== Cadenas vacias por columna (texto) ===")
for c in df.select_dtypes(include="object").columns:
    vacios = (df[c].astype(str).str.strip() == "").sum()
    if vacios > 0:
        print(f"{c}: {vacios} vacios")
print("(TotalCharges ya no esta en el dataframe: se elimino en el Paso 1, junto con sus 7 valores vacios.)")

print("\n=== Duplicados (en las columnas que va a ver el modelo) ===")
dup_mask = df.duplicated(keep=False)
print("Filas involucradas en algun grupo de duplicados:", dup_mask.sum())
print("Grupos distintos de filas identicas:", df[dup_mask].drop_duplicates().shape[0])
print("Distribucion de tenure en esas filas:")
print(df.loc[dup_mask, "tenure"].value_counts().sort_index())

print("\n=== Atipicos (IQR, 1.5x) en variables numericas ===")
for c in ["tenure", "MonthlyCharges"]:
    s = df[c]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((s < lo) | (s > hi)).sum()
    print(f"{c}: rango=[{s.min()}, {s.max()}]  limites_IQR=[{lo:.2f}, {hi:.2f}]  atipicos={n_out}")

### Hallazgos y decisiones

- **Faltantes:** fuera de `TotalCharges` (ya excluida por fuga de información), **no hay ningún valor faltante** en las 17 columnas restantes: 0 `NaN` declarados y 0 cadenas vacías. No queda nada por imputar en el sentido estricto — ver la comparación de estrategias en el Paso 5, que se mantiene como medida defensiva del pipeline.
- **Duplicados:** en las 17 columnas que va a ver el modelo (tras quitar `TotalCharges`, `customerID` y `gender`), 42 filas (20 grupos) coinciden exactamente con al menos otra. El 81 % de esas filas (34 de 42) tiene `tenure = 1` (clientes en su primer mes, donde el catálogo de combinaciones de plan es pequeño: solo 316 combinaciones distintas de precio/contrato/internet/pago entre 388 clientes con `tenure = 1`, así que coincidencias exactas son estadísticamente esperables). El resto son un puñado de clientes de tenure más alto (2, 25 y 69 meses) que comparten el plan más barato posible (sin internet, solo teléfono) — la esquina de menor variedad del catálogo, donde volver a coincidir tampoco sorprende. Se investigaron antes de decidir (ver README, sección "Duplicados"): cada fila tiene un `customerID` original único y no son filas adyacentes en el archivo, lo que descarta un error de copiar-pegar. **Decisión: se conservan todas las filas** — coincidir en variables observables no implica ser el mismo registro capturado dos veces.
- **Atípicos:** el método IQR no encontró ningún valor atípico en `tenure` ni `MonthlyCharges`; los rangos observados (0–72 meses, USD 18.40–118.75) son consistentes con límites de negocio razonables. **Decisión: no se aplica ningún recorte (capping).**